# Phase 4 — a group is more than its average

Rossmann v7 spent half its run re-drilling one hole.

| table | probe | gain | what it did |
|---|---|---|---|
| t7 | 0.1587 | | built `Store_DOW = Store*10 + DayOfWeek` |
| **t8** | **0.1344** | **+0.0243** | target-encoded it — "average sales for this store on a Monday" |
| t9–t15 | 0.1342 | **+0.0006 total** | nine more composite keys, same tool, three negative |

`t8` was the biggest late gain of the run. Then nine more tables bought 2.5% of
the return between them.

That is not the agent being weak. Every per-group feature it could build went
through **one** mechanism — target-encode a hand-made composite key — and once
`t8` existed there was nothing else the harness let it *say* about a group. It
exhausted its vocabulary and then repeated itself.

Without the dataset in the sentence: **the harness could learn a group's average
target and nothing else about that group's target distribution.**

This notebook drives the fix by hand. Decision record:
[0046](../../../research/archive/decisions/0046-the-mean-is-a-lossy-summary-of-a-group.md).

## 1. Two categories the mean cannot tell apart

Nothing subtle here — `tight` and `wide` are built to share a target mean and
differ in everything else.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)


def make_panel(n_per=400):
    """Four categories. `tight` and `wide` have the SAME mean target."""
    rows = []
    for cat, mean, spread in [
        ("tight", 50.0, 1.0),
        ("wide", 50.0, 25.0),
        ("low", 20.0, 5.0),
        ("high", 90.0, 5.0),
    ]:
        y = rng.normal(mean, spread, size=n_per)
        rows.append(pd.DataFrame({"cat": cat, "y": y}))
    return pd.concat(rows).sample(frac=1.0, random_state=0).reset_index(drop=True)


panel = make_panel()
print(panel.groupby("cat")["y"].agg(["mean", "std", "count"]).round(2))
print("\n`tight` and `wide` are the same number to a target encoder.")

In [ ]:
from tabular_agent.encoders import TargetEncoder

mean_only = TargetEncoder(["cat"], smoothing=0.0).fit(panel, panel["y"])
print(mean_only.transform(pd.DataFrame({"cat": ["tight", "wide"]})).round(2))

Two rows, indistinguishable. A model handed `cat_te` has no way to know that one
of these categories is predictable to ±1 and the other to ±25.

`stats=` asks for the rest of the distribution. The mean keeps its bare
`_te` name, so nothing written before this existed changes.

In [ ]:
with_spread = TargetEncoder(
    ["cat"], smoothing=0.0, stats=["mean", "std", "median", "count"]
).fit(panel, panel["y"])

print(with_spread.output_columns)
print(with_spread.transform(pd.DataFrame({"cat": ["tight", "wide"]})).round(2))

## 2. Why this needed a tool at all

The obvious way to write it is one line of pandas. Try it in the free-code path.

In [ ]:
from tabular_agent.transform import transform_features

CODE = """
def transform_features(df):
    df = df.copy()
    df["cat_spread"] = df.groupby("cat")["y"].transform("std")
    return df
"""

try:
    transform_features(panel, CODE)
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:260]}...")

That refusal is 0011 and it is **right**. Only the *code* is stored, so at
predict time it recomputes the spread from whatever rows arrived in that request
— one row alone would get `NaN`, a batch of a million something else again. The
same row scores differently depending on what it was batched with.

So why is the encoder legal doing the same arithmetic? Because **the number is
fitted once and stored**. The lookup reads the row's category and nothing else,
so a single row maps to exactly what a million rows would. 0011 would pass it if
it could see it.

The other near-miss is `fit_group_stats` (0032), which already has the whole
`mean/median/std/count` menu. Mechanically it would compute this fine:

In [ ]:
from tabular_agent.encoders import GroupStatsEncoder

enc = GroupStatsEncoder(["cat"], "y", "std").fit(panel)
print("mechanically fine:", enc.output_column, "->", round(enc.mapping_["wide"], 2))
print("\n...but the *tool* refuses the target as its value column. Section 5 runs it.")

The refusal is deliberate: a per-group statistic of the target *is* target
encoding, and it needs smoothing and out-of-fold fitting that `fit_group_stats`
does not do. The capability was blocked from both directions, correctly from
both — which is exactly why it had to be built rather than worked around.

## 3. What each statistic buys, and why only the mean is smoothed

The median is not a spare mean. Give one category a single huge value:

In [ ]:
skewed = pd.DataFrame(
    {"cat": ["s"] * 3 + ["t"] * 3, "y": [1.0, 2.0, 300.0, 90.0, 100.0, 110.0]}
)
enc = TargetEncoder(["cat"], smoothing=0.0, stats=["mean", "median"]).fit(
    skewed, skewed["y"]
)
print(enc.transform(pd.DataFrame({"cat": ["s", "t"]})).round(2))

By mean, `s` and `t` are near-twins (101 vs 100). By median they are 2 and 100.

Now smoothing. It pulls a thin category's estimate toward the population value in
proportion to how little evidence stands behind it. Watch which columns move for
a category with **three rows**:

In [ ]:
STATS = ["mean", "std", "median", "count"]
thin = pd.concat([panel, pd.DataFrame({"cat": ["rare"] * 3, "y": [95.0, 96.0, 97.0]})])

for smoothing in [0.0, 500.0]:
    enc = TargetEncoder(["cat"], smoothing=smoothing, stats=STATS).fit(thin, thin["y"])
    row = enc.transform(pd.DataFrame({"cat": ["rare"]})).round(2).to_dict("records")[0]
    print(f"smoothing={smoothing:<6} {row}")

**Only `cat_te` moved.** That asymmetry is a decision, not an oversight.

The weighted-average formula that does smoothing is a statement about *means*.
Shrinking a median toward the global median by a count weight, or a standard
deviation toward the global standard deviation, would be inventing an estimator
and giving it the name of a standard one.

What replaces it is better anyway: **`cat_te_count` is right there**, and says
`3.0`. The model learns how far to trust a thin category instead of having a
fixed answer baked in by a smoothing constant nobody tuned. Same information,
expressed rather than assumed.

The menu is four entries. `min`/`max` are off it because an extreme of the target
within a group is essentially *one row's target*, and it survives out-of-fold
fitting only by the luck of which fold that row landed in. `sum` is off it
because it is exactly `count * mean`.

## 4. Leakage — and a test that would have passed while the leak was real

Every one of these reads `y`, so every one needs the out-of-fold discipline.

The existing leak test for the mean correlates the encoding with `y` and shows
the naive version fabricates signal. **Applied to `std` that test fails — and not
because `std` is safe.** A row inflates its group's spread whether its own target
is unusually high *or* unusually low, so the leak lands on `|y - mean|` and can
sit near zero against `y` itself.

Correlation is the wrong probe for a statistic that is not monotonic in the row's
own value. So test the property directly: **move one row's target, watch that
row's own feature.**

In [ ]:
from tabular_agent.cv import make_folds
from tabular_agent.encoders import oof_target_encode

r = np.random.default_rng(7)
small = pd.DataFrame({"cat": [f"k{i % 60}" for i in range(300)]})  # ~5 rows each
y = r.normal(size=300)
moved = y.copy()
moved[0] += 50.0  # one row, far away

folds = make_folds(y, "regression", n_splits=5, seed=42)


def naive(target):
    return TargetEncoder(["cat"], 0.3, STATS).fit(small, target).transform(small)


def out_of_fold(target):
    frame, _ = oof_target_encode(small, target, ["cat"], folds, 0.3, STATS)
    return frame


print(f"{'column':16s} {'fitted on all rows':>24s}   {'out-of-fold':>22s}")
for name in ["cat_te", "cat_te_std", "cat_te_median"]:
    n0, n1 = naive(y)[name].iloc[0], naive(moved)[name].iloc[0]
    o0, o1 = out_of_fold(y)[name].iloc[0], out_of_fold(moved)[name].iloc[0]
    print(f"{name:16s} {n0:+10.3f} -> {n1:+10.3f}   {o0:+9.3f} -> {o1:+9.3f}")

Fitted on all rows, row 0 moves its own feature — through the mean, through the
spread, and through the median. That is the leak, and out-of-fold it is gone: row
0 is encoded by folds it was not in, so its own answer cannot reach it.

One mechanism, every statistic, and it asserts the thing that actually matters
rather than a proxy that happens to work for means.

The generalisable lesson: **a leak test written for one statistic does not
transfer to another statistic of the same column.**

`count` moves under neither, and that is correct rather than a gap — it never
reads the target's *value*, only that a row was there. It is still fitted
out-of-fold, because a second looser path through the same tool is something
somebody has to keep correct forever in exchange for nothing measurable.

## 5. Through the harness

Same thing as a run would do it. Note `serving_columns`: `y` is not in a
prediction request, which is what makes the next-to-last cell the one worth
watching.

In [ ]:
import os
import tempfile

from tabular_agent.harness.session import RunSession
from tabular_agent.harness.tools import fit_encoding, inspect_data, train_tabular_model
from tabular_agent.taskspec import TaskSpec

tmp = tempfile.mkdtemp()
data_path = os.path.join(tmp, "panel.csv")
panel.assign(noise=rng.normal(size=len(panel))).to_csv(data_path, index=False)

session = RunSession(
    os.path.join(tmp, "run"),
    data_path,
    TaskSpec(target="y", task_type="regression", metric="rmse", n_splits=3),
    serving_columns=["cat", "noise"],
)
inspect_data(session)

out = fit_encoding(session, columns=["cat"], method="target", stats=STATS)
print(out["new_columns"])
print("probe:", out.get("probe_score"))

The two refusals, live. First the one section 2 promised — `fit_group_stats`
pointed at the target:

In [ ]:
from tabular_agent.harness.tools import fit_group_stats

for call in [
    lambda: fit_group_stats(session, by=["cat"], value="y", stat="std"),
    lambda: fit_encoding(session, columns=["cat"], method="frequency", stats=["count"]),
]:
    try:
        call()
    except ValueError as e:
        print(f"ValueError: {e}\n")

Both point at the tool that *does* the job rather than just saying no — the
refusal message is the whole channel through which an agent learns the harness's
shape (0032's `filter_rows` was used unprompted on nothing but its docstring).

Now the failure that would otherwise be silent. **Every one of these four columns
was computed from `y`, and `y` is not in a prediction request.** They survive only
because the fitted statistics live in the table's lineage and the lookup reads
`cat` alone:

In [ ]:
from tabular_agent.harness.tools import finalize_run
from tabular_agent.io import load_table
from tabular_agent.pipeline import load_bundle, predict_bundle

train_tabular_model(session, table=out["table"], model_type="lgbm")
finalize_run(session)

bundle = load_bundle(os.path.dirname(session.ctx.path("bundle", "_")))
serving_rows = load_table(data_path).drop(columns=["y"]).head(5)
print(serving_rows)
print("\npredictions:", np.round(predict_bundle(bundle, serving_rows), 2))

## 6. What this does not close

This was **slice 1 of three** in `docs/time_aware_features.md`. It closes Gap A —
only-the-mean. Gap B is untouched and is the larger half:

> **No feature can look at another row in time.** No lag, no rolling mean, no
> days-since-last-event. On a forecasting problem that is most of the signal.

It is structural, not a missing convenience. `transform_features` *could* write
`groupby().shift()` and 0011 would kill it for the same reason it killed the
spread above — and at predict time the served rows are later than every training
row, so the lag has to be looked up from history the transform does not have.
**A lag needs the harness to carry a time series across the fit/predict seam, and
it can only carry a mapping.**

The two questions that blocked that build are now answered in
[0047](../../../research/archive/decisions/0047-the-forecast-horizon-is-a-column-not-a-constant.md):
history is frozen at the end of training, each served row gets the last known
value plus a **per-row staleness** column, and no served row reads its own
batch's past. The row at `T+1` gets staleness 1 — which *is* "yesterday's value"
— and the row at `T+5` gets the same value with staleness 5, so the feature
degrades visibly instead of lying quietly.

And the honest caveat on all of it: **no run has used any of this yet.** 0032 is
the cautionary precedent — `fit_group_stats` was the tool the agent explicitly
asked for, and then never called once in 69 tool calls.